# Final Spring 2026

## Project Overview

You have been hired by a data science company to complete a specific analytical task. Your responsibility is to complete the task and present your results in a notebook.

**Important:** Your audience (stakeholders, managers, and decision-makers) is not interested in code implementation details or technical minutiae. They want to understand:

- **Your findings** - What insights did you discover?
- **Your solution** - What approach did you take and why?
- **The results** - What are the outcomes and recommendations?

Focus on clear, concise explanations of your analysis, methodology, and conclusions. Use visualizations and summaries to communicate effectively. The code should support your narrative, not dominate it.

**Rubric** https://docs.google.com/spreadsheets/d/1H5GNS4BRGbaB3YCEFMocpSBDT2-XpZsB/edit?usp=sharing&ouid=114674526770118608358&rtpof=true&sd=true

# Option1

In [1]:
from IPython.display import Markdown, display
display(Markdown("fake_profile_dataset_documentation.md"))

# Fake Profile Detection

## Business Context

**SocialTrust Analytics** is a social media trust and safety analytics company hired by online platforms to identify accounts that may be fake, spam-oriented, or suspicious. The goal is to help moderation teams prioritize accounts for review while still giving analysts enough information to explain why an account looks risky.

This dataset does NOT include the target variable in the main student file. The data scientist must analyze the unlabeled profile data, engineer useful features, discover natural account groups, and assign likely fake-profile labels before checking the separate evaluation labels.

## Dataset Overview

- **Total Records**: 5,000
- **Features**: 19 
- **Target Variable**: NOT INCLUDED 
- **Missing Values**: 0
- **Class Distribution in Labels File**: 3,198 normal profiles and 1,802 fake profiles
- **Fake Profile Rate**: 36.04%

## Feature Categories

### 1. Account Profile Features
- `profile_id`: Unique profile identifier
- `account_age_days`: Age of the account in days
- `bio_length`: Number of characters in the profile bio
- `profile_picture_quality`: Profile image quality score from 0 to 1
- `external_link_count`: Number of external links in the profile
- `username_digit_count`: Number of digits in the username
- `username_length`: Username length

### 2. Audience and Network Features
- `followers_count`: Number of followers
- `following_count`: Number of accounts followed

### 3. Posting Activity Features
- `posts_count`: Lifetime number of posts
- `posts_last_30_days`: Number of posts made in the last 30 days
- `recent_posts_sampled`: Number of recent posts reviewed for behavior signals
- `avg_caption_length`: Average caption length for sampled recent posts
- `repeated_captions`: Number of repeated captions in sampled recent posts
- `night_posts`: Number of sampled posts made during nighttime hours

### 4. Engagement Features
- `likes_last_20_posts`: Total likes across the most recent 20 posts
- `comments_last_20_posts`: Total comments across the most recent 20 posts

### 5. Direct Message Features
- `dm_sent_last_7_days`: Number of direct messages sent in the last 7 days
- `dm_replies_last_7_days`: Number of direct message replies received in the last 7 days

### 6. Evaluation Target
- `fake_profile`: Hidden binary target where 1 means the account behaves like a fake or suspicious profile and 0 means the account behaves like a normal profile

## Important Feature Engineering Ideas

The raw columns are intentionally incomplete on their own. Students should create ratios, rates, and interaction features before clustering or modeling.

## Expected Profile Patterns

### 1. Normal Established Profiles
Older accounts with better profile picture quality, longer bios, healthier follower/following ratios, lower duplicate caption rates, and stronger comment quality. These profiles are usually `fake_profile = 0`.

### 2. Suspicious Growth or Spam Profiles
Newer accounts with high following counts, weak follower/following ratios, more digits in usernames, repeated captions, high night posting, and high direct message pressure. These profiles are usually `fake_profile = 1`.

### 3. Ambiguous Creator or Brand-Like Profiles
Some real profiles may post often, include external links, or follow many accounts. These cases make the clusters overlap and require students to justify labeling decisions instead of relying on a single feature.

## Using the Subject Matter Expert (SME)

The SME class allows you to query label for a specific `profile_id`. It should be treated like a limited expert resource rather than as a dataset to inspect directly.

### Example Usage:

```python
from fake_profile_sme import SME

# Initialize SME
sme = SME()

fake_profile = sme.ask("P00001")

print(f"Fake profile: {fake_profile}")
```

### Important Notes:
- You can ask up to 500 times
- Pass one `profile_id` string to `sme.ask()`
- `sme.ask(profile_id)` returns `0` for a likely normal profile and `1` for a fake or suspicious profile
- Do not use `fake_profiles_labels.csv` during the initial clustering and label creation process
- Scale numeric features before using distance-based clustering methods
- Inspect cluster centers and summary statistics before assigning labels
- Ambiguous clusters are expected and should be discussed in the analysis
- Compare student-created labels against `fake_profile` only at the end, after the semi-supervised workflow is complete


In [2]:
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/msaricaumbc/DS_data/master/ds602/final/fake_profiles_unlabeled.csv')
df.shape

(5000, 19)

In [3]:
df.head().T

,0,1,2,3,4
profile_id,P00001,P00002,P00003,P00004,P00005
account_age_days,549,204,1021,998,1096
bio_length,131,29,73,7,28
profile_picture_quality,0.575,0.319,0.572,0.726,0.825
posts_count,350,30,374,127,287
followers_count,1681,317,312,169,336
following_count,60,1701,451,192,598
posts_last_30_days,0,33,11,5,8
external_link_count,0,1,0,0,0
username_digit_count,0,1,2,2,1


# Option2

In [4]:
display(Markdown("student_success_dataset_documentation.md"))

# Option1

## Student Success Prediction

### Business Context

**Campus Success Analytics** is helping an introductory course team identify students who may need support before the end of the semester. The goal is to use course activity, assessment progress, and workload indicators to discover meaningful student groups, create tentative labels, and then build a supervised model that predicts likely course success.

This is a **semi-supervised classification** task. You should begin with the unlabeled data, use feature engineering and clustering to create your own student success labels, and only use the provided labels file at the end for evaluation.

### Dataset Overview

- **Total Records**: 5,000
- **Features**: 17 modeling features
- **Target Variable**: NOT INCLUDED in the unlabeled file
- **Missing Values**: 0
- **Success Rate**: 56.82%
- **Class Distribution**: {0: 2159, 1: 2841}

### Feature Categories

#### 1. Attendance Features
- `classes_attended`: Number of class meetings the student attended
- `classes_total`: Total number of class meetings available

#### 2. Study and Engagement Features
- `study_hours_per_week`: Estimated weekly study time
- `discussion_posts`: Number of course discussion posts
- `office_hours_visits`: Number of office hours visits
- `lms_logins_last_30_days`: Learning management system logins in the last 30 days

#### 3. Assignment Features
- `assignments_submitted`: Number of assignments submitted
- `assignments_total`: Total number of assignments available
- `late_submissions`: Number of late assignments

#### 4. Assessment Features
- `quiz_points_earned`: Quiz points earned so far
- `quiz_points_possible`: Quiz points possible so far
- `practice_exam_points`: Practice exam points earned
- `practice_exam_possible`: Practice exam points possible
- `prior_gpa`: Student's prior GPA

#### 5. Life and Workload Features
- `sleep_hours`: Average nightly sleep hours
- `work_hours_per_week`: Weekly outside work hours
- `commute_minutes`: Typical commute time in minutes

#### 6. Target
- `likely_success`: 1 if the student is likely to pass or succeed, 0 if the student may be at risk

### Using the Subject Matter Expert (SME)

The SME class allows you to query label for a specific `student_id`. It should be treated like a limited expert resource rather than as a dataset to inspect directly.

#### Example Usage:

```python
from student_success_sme import SME

# Initialize SME
sme = SME()

# Query the hidden label for one student_id
likely_success = sme.ask("S00001")

print(f"Likely success: {likely_success}")
```

#### Important Notes:
- You can ask up to 500 times
- Pass one `student_id` string to `sme.ask()`
- `sme.ask(student_id)` returns `0` for an at-risk student and `1` for a student likely to succeed
- The `student_id` must exist in the student dataset
- Do not use `student_success_labels.csv` during the initial clustering and label creation process
- Scale numeric features before using distance-based clustering methods
- Inspect cluster centers and summary statistics before assigning labels
- Ambiguous clusters are expected and should be discussed in the analysis
- Compare student-created labels against `likely_success` only at the end, after the semi-supervised workflow is complete



In [5]:
df = pd.read_csv('https://raw.githubusercontent.com/msaricaumbc/DS_data/master/ds602/final/student_success_unlabeled.csv')
df.shape

(5000, 18)

In [6]:
df.head().T

,0,1,2,3,4
student_id,S00001,S00002,S00003,S00004,S00005
classes_attended,18,22,10,23,26
classes_total,30,26,28,24,26
study_hours_per_week,5.9,14.0,5.3,9.8,13.0
assignments_submitted,11,10,6,11,10
assignments_total,13,10,13,13,11
quiz_points_earned,65,69,53,64,59
quiz_points_possible,90,90,90,110,90
discussion_posts,4,6,8,8,7
office_hours_visits,3,4,1,6,3


# Option3

In [7]:
from IPython.display import Markdown, display
display(Markdown("ecommerce_dataset_documentation.md"))


# E-commerce Customer Purchase Prediction

## Business Context

**Company A Solutions** is an e-commerce analytics company hired by online retailers to predict which customers are likely to make a purchase during their browsing session. This helps optimize marketing spend and personalize user experience in real-time.

This dataset does NOT include the target variable. The data scientist must analyze the data and ask the subject matter expert if a customer will purchase for a limited number of times.

## Dataset Overview

- **Total Records**: 50,000
- **Features**: 26
- **Target Variable**: NOT INCLUDED
- **Missing Values**: 10000

## Feature Categories

### 1. Temporal Features
- `session_start_time`: Timestamp when session began
- `hour`: Hour of day (0-23)
- `day_of_week`: Day of week (0=Monday, 6=Sunday)
- `month`: Month (1-12)
- `is_weekend`: Binary flag for weekend (1) vs weekday (0)
- `session_duration_minutes`: Length of session in minutes
- `days_since_last_visit`: Days since customer's last visit

### 2. Behavioral Numerical Features
- `page_views`: Number of pages viewed during session
- `clicks`: Number of clicks made during session
- `avg_scroll_depth`: Average scroll depth (0-1)
- `items_added_to_cart`: Number of items added to cart
- `items_removed_from_cart`: Number of items removed from cart
- `search_queries_count`: Number of search queries made
- `product_page_time_minutes`: Time spent on product pages
- `categories_viewed_count`: Number of different categories viewed
- `avg_price_viewed`: Average price of products viewed ($)

### 3. Categorical Features
- `device_type`: mobile, desktop, tablet
- `traffic_source`: organic_search, paid_search, social_media, email, direct, referral
- `customer_segment`: new_customer, returning_customer, vip_customer, at_risk_customer
- `region`: North_America, Europe, Asia_Pacific, Latin_America, Middle_East_Africa
- `browser`: Chrome, Safari, Firefox, Edge, Other
- `operating_system`: Windows, macOS, iOS, Android, Linux
- `user_type`: new_user, returning_user

### 4. Text Features
- `search_queries`: Comma-separated search terms used
- `categories_viewed`: Comma-separated product categories viewed
- `user_agent`: Browser user agent string

## Using the Subject Matter Expert (SME)

The SME class allows you to query purchase probabilities for specific customer profiles.

### Example Usage:

```python
from ecommerce_sme import SME

# Initialize SME
sme = SME()

# Query purchase by passing index
purchase_prob = sme.ask(10)

print(f"Purchase probability: {purchase_prob}")
```

### Important Notes:
- You can ask up to 500 times (SME has other tasks.)
- If no matching records found, SME will raise an exception


In [8]:
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/msaricaumbc/DS_data/master/ds602/final/ecommerce_sessions_X.csv')
df.shape

(50000, 26)

In [9]:
df.head().T

,0,1,2,3,4
session_start_time,2024-08-21 13:28:14,2024-01-10 19:20:38,2024-04-10 14:22:10,2024-03-22 14:52:35,2024-09-25 09:02:21
hour,13,19,14,14,9
day_of_week,2,2,2,4,2
month,8,1,4,3,9
is_weekend,0,0,0,0,0
session_duration_minutes,17.323063,3.268847,16.22868,4.920244,13.404538
days_since_last_visit,3.044042,11.887877,0.264852,13.660919,0.30211
page_views,6.132345,0.884203,3.060268,1.014015,3.821892
clicks,2.069694,0.0,1.042415,0.967902,2.012747
avg_scroll_depth,0.261632,0.590109,0.223406,0.403105,0.310772


# Option4

In [10]:
display(Markdown("churn_dataset_documentation.md"))


# Streaming Service Customer Churn Prediction

## Business Context

**StreamFlix** is a popular streaming service that wants to identify customers who are likely to cancel their subscription within the next 30 days. Early identification allows the company to implement targeted retention strategies, such as personalized content recommendations, promotional offers, or customer support outreach, to reduce churn and maintain revenue.

This dataset does NOT include the target variable. The data scientist must analyze the data and ask the subject matter expert if a customer will churn for a limited number of times. 

## Dataset Overview

- **Total Records**: 50,000
- **Features**: 37
- **Target Variable**: NOT INCLUDED

- **Missing Values**: 12500

## Feature Categories

### 1. Subscription Features
- `subscription_start_date`: Date when customer first subscribed
- `subscription_length_days`: Number of days since subscription started
- `subscription_length_months`: Number of months since subscription started
- `subscription_plan`: basic, standard, premium, family, student
- `monthly_price`: Monthly subscription price ($)
- `billing_cycle`: monthly, annual
- `payment_method`: credit_card, debit_card, paypal, apple_pay, google_pay, bank_transfer

### 2. Usage Behavioral Features (Last 30 Days)
- `total_watch_time_hours`: Total hours of content watched
- `sessions_count`: Number of viewing sessions
- `avg_session_duration_minutes`: Average length of viewing sessions
- `days_since_last_watch`: Days since customer last watched content
- `unique_titles_watched`: Number of different titles watched
- `content_completion_rate`: Percentage of started content that was completed
- `abandoned_series_count`: Number of series started but not finished
- `binge_sessions_count`: Number of binge-watching sessions (>3 hours)

### 3. Content Preference Features
- `primary_genre`: Most watched genre (Action, Comedy, Drama, etc.)
- `genres_watched_count`: Number of different genres watched
- `genres_watched`: Comma-separated list of genres watched
- `content_type_preference`: movies, tv_shows, both
- `new_content_ratio`: Ratio of new releases vs catalog content watched

### 4. Device and Platform Features
- `primary_device`: smart_tv, mobile, tablet, desktop, game_console, streaming_device
- `devices_used_count`: Number of different devices used
- `devices_used`: Comma-separated list of devices used
- `primary_os`: Operating system of primary device

### 5. Engagement and Interaction Features
- `has_profile`: Binary flag for profile creation (1) or not (0)
- `number_of_profiles`: Number of user profiles on account
- `watchlist_size`: Number of titles in watchlist
- `ratings_given_count`: Number of content ratings provided
- `reviews_written_count`: Number of reviews written
- `downloads_count`: Number of titles downloaded for offline viewing

### 6. Support and Service Features
- `support_tickets_count`: Number of support tickets opened (last 90 days)
- `support_ticket_reasons`: Comma-separated reasons for support tickets
- `payment_failures_count`: Number of payment failures (last 90 days)
- `account_on_hold`: Binary flag for account suspension (1) or not (0)
- `used_free_trial`: Binary flag if customer used free trial before subscribing

### 7. Temporal Features
- `current_month`: Current month (1-12) for seasonality analysis
- `days_until_next_billing`: Days until next billing date

## Using the Subject Matter Expert (SME)

The SME class allows you to query churn for specific customer profiles.

### Example Usage:

```python
from streamflix_sme import SME

# Initialize SME
sme = SME()

# Query churn
# Pass index 
churn = sme.ask(12)

print(f"Churn: {churn}")
```

### Important Notes:
- You can ask up to 500 times (sme has other tasks.)
- If no matching records found, SME will raise an exception


In [11]:
df = pd.read_csv('https://raw.githubusercontent.com/msaricaumbc/DS_data/master/ds602/final/streaming_churn_dataset.csv')
df.shape

(50000, 37)

In [12]:
df.head().T

,0,1,2,3,4
subscription_start_date,2022-11-07,2023-02-28,2024-01-31,2021-05-21,2023-04-05
subscription_length_days,785,672,335,1320,636
subscription_plan,basic,standard,standard,standard,family
monthly_price,9.99,15.99,15.99,15.99,24.99
billing_cycle,monthly,monthly,monthly,monthly,annual
payment_method,credit_card,apple_pay,debit_card,credit_card,credit_card
total_watch_time_hours,74.963172,13.848148,43.368652,28.69851,204.641244
sessions_count,29.742333,8.341995,20.970448,13.858491,95.626485
avg_session_duration_minutes,152.370633,102.980411,120.414698,112.382035,119.821246
days_since_last_watch,5.58705,5.350973,10.167759,20.525603,6.219125


# Production Notebook

In [13]:
# don't use this
from random import randint

class TestModel:
    def predict(self, X):
        return [ randint(0, 1) for _ in range(len(X))]

In [18]:
# In the second notebook

from sklearn.metrics import classification_report

def production(X_path, y_path):
    # load your model
    model = TestModel()
    
    # load data
    df_X = pd.read_csv(X_path)

    # make the changes if required 
    # -------------------------

    

    # -------------------------
    pred = model.predict(df_X)

    df_Y = pd.read_csv(y_path)
    y_label = df_Y.columns[-1]
    # print(y_label)
    df_y = df_Y[y_label]
    print(classification_report(df_y, pred))
    

production( 
    X_path='https://raw.githubusercontent.com/msaricaumbc/DS_data/master/ds602/final/student_success_unlabeled.csv',
    y_path='https://raw.githubusercontent.com/msaricaumbc/DS_data/master/ds602/final/student_success_labels.csv'
)
    

              precision    recall  f1-score   support

           0       0.43      0.50      0.46      2159
           1       0.57      0.50      0.53      2841

    accuracy                           0.50      5000
   macro avg       0.50      0.50      0.50      5000
weighted avg       0.51      0.50      0.50      5000

